# Module 7 Day 2 — ArgumentCorrectnessMetric + StepEfficiencyMetric + Coverage Matrix

**Module 7 · Session 2 of 2**

Day 1 caught wrong-tool failures (process_refund called for a status check) and no-tool failures (the Air Canada pattern). Day 2 goes one layer deeper:

1. **`ArgumentCorrectnessMetric`** — right tool, wrong argument (e.g. order_id contaminated from context)
2. **`StepEfficiencyMetric`** — correct result, too many steps to get there
3. **Full coverage matrix** — four modules of failure modes, now formalized
4. **Retiring Module 6's hand-rolled judges** — replace them with confidence

---

**Module 4 Day 4 connections this session makes explicit:**
- Boundary value analysis → the `order_id` boundary between current-message vs. context-history is a BVA problem
- Coverage matrix → extended with two new columns from today's metrics
- Hard negatives → `refund-01-hardneg` (argument contamination) and `escalation-01-hardneg` (inefficiency) are today's new hard negatives

## Setup

In [1]:
import asyncio
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from agent_tools import run_agent, to_deepeval_tool_calls
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import (
    TaskCompletionMetric,
    ToolCorrectnessMetric,
    ArgumentCorrectnessMetric,
    StepEfficiencyMetric,
)

provider = os.getenv("PROVIDER", "azure")
print(f"PROVIDER={provider}")

tracing = os.getenv("LANGSMITH_TRACING", "false").lower() == "true"
project = os.getenv("LANGSMITH_PROJECT", "module-07-deepeval-agent-testing")
if tracing:
    print(f"LangSmith tracing enabled — project: {project}")
else:
    print("LangSmith tracing disabled (set LANGSMITH_TRACING=true to enable)")

print("All 4 DeepEval metrics imported successfully.")

PROVIDER=azure
LangSmith tracing enabled — project: module-07-deepeval-agent-testing
All 4 DeepEval metrics imported successfully.


## Part 1 — ArgumentCorrectnessMetric: right tool, wrong order_id

This is the failure mode Module 6's `MemoryRetentionVerdict` was approximating. The question is no longer "did the correct fact survive into the final answer" (retrieval world) but "did the agent pass the right value to the tool" (tool-calling world).

**Module 4 Day 4 BVA connection:** the `order_id` sits at a boundary: the value in the *current* user message vs. any value that appeared elsewhere in the context. BVA tells you to write test cases at that boundary — specifically, a case where the contaminating value is a real, valid order_id (so the tool succeeds and the output looks plausible, making it genuinely hard to detect).

In [2]:
argument_metric = ArgumentCorrectnessMetric(threshold=0.7, verbose_mode=True)

# Happy path: correct order_id extracted from the message
refund_happy = LLMTestCase(
    input="I'd like to request a refund for order #67890. The product arrived damaged.",
    actual_output="I've initiated a refund for order #67890 (TurboMax Pro) in the amount of $299.00. Your refund ID is R-XY9901.",
    tools_called=[
        ToolCall(
            name="process_refund",
            input_parameters={"order_id": "67890", "reason": "product arrived damaged"},
            output='{"refund_id": "R-XY9901", "amount": 299.0, "status": "initiated", "reason": "product arrived damaged"}',
        )
    ],
)

# Hard negative: contaminated order_id (12345 was from a previous turn, not the current request)
refund_wrong_arg = LLMTestCase(
    input="I'd like to request a refund for order #67890. The product arrived damaged.",
    # Output says $199 (order 12345's amount) — the tell that the wrong order was refunded
    actual_output="I've initiated a refund for your order in the amount of $199.00. Your refund ID is R-ZZ8812.",
    tools_called=[
        ToolCall(
            name="process_refund",             # CORRECT tool
            input_parameters={
                "order_id": "12345",           # WRONG — contaminated from earlier context
                "reason": "product arrived damaged",
            },
            output='{"refund_id": "R-ZZ8812", "amount": 199.0, "status": "initiated", "reason": "product arrived damaged"}',
        )
    ],
)

print("=== HAPPY PATH: correct order_id ===")
argument_metric.measure(refund_happy)
print("ArgumentCorrectnessMetric:")
print(f"  Score: {argument_metric.score:.2f} | Passed: {argument_metric.is_successful()}")
print(f"  Reason: {argument_metric.reason}")
print()

print("=== HARD NEGATIVE: contaminated order_id from previous context ===")
argument_metric.measure(refund_wrong_arg)
print("ArgumentCorrectnessMetric:")
print(f"  Score: {argument_metric.score:.2f} | Passed: {argument_metric.is_successful()}")
print(f"  Reason: {argument_metric.reason}")
print()

print("This is the argument contamination bug:")
print("  - User asked about order #67890")
print("  - Agent called process_refund(order_id='12345') — pulled from earlier context")
print("  - The tool succeeded (12345 is a real order!) making this hard to detect from output alone")
print("  - The output '$199.00' (order 12345's amount) vs '$299.00' (order 67890's amount) is the tell")
print("  - ToolCorrectnessMetric would PASS this (right function), only ArgumentCorrectnessMetric catches it")

=== HAPPY PATH: correct order_id ===
ArgumentCorrectnessMetric:
  Score: 0.95 | Passed: True
  Reason: The agent correctly extracted order_id='67890' from the user's message and passed it to process_refund. The argument matches the order referenced in the customer's request.

=== HARD NEGATIVE: contaminated order_id from previous context ===
ArgumentCorrectnessMetric:
  Score: 0.08 | Passed: False
  Reason: The agent passed order_id='12345' to process_refund, but the user's message clearly referenced order #67890. The argument appears to be contaminated from a previous context turn. The refund would be applied to the wrong order.

This is the argument contamination bug:
  - User asked about order #67890
  - Agent called process_refund(order_id='12345') — pulled from earlier context
  - The tool succeeded (12345 is a real order!) making this hard to detect from output alone
  - The output '$199.00' (order 12345's amount) vs '$299.00' (order 67890's amount) is the tell
  - ToolCorrectnes

## Part 2 — Verify: ToolCorrectnessMetric passes the wrong-argument case

This is the key demonstration of why you need `ArgumentCorrectnessMetric` in addition to `ToolCorrectnessMetric`. Wrong-arg is invisible to wrong-tool detection.

In [3]:
tool_metric = ToolCorrectnessMetric(threshold=0.7)

print("Running ToolCorrectnessMetric on the WRONG ARGUMENT case...")
tool_metric.measure(refund_wrong_arg)
print(f"  Score: {tool_metric.score:.2f} | Passed: {tool_metric.is_successful()}")
print(f"  Reason: {tool_metric.reason}")
print()

if tool_metric.is_successful():
    print("ToolCorrectnessMetric PASSED the wrong-argument case (as expected).")
    print("This confirms: wrong-tool and wrong-argument are distinct failures requiring distinct metrics.")
else:
    print("ToolCorrectnessMetric failed the wrong-argument case — this is unexpected. Review the metric behavior.")

print()
print("Now running ArgumentCorrectnessMetric on the same case...")
arg_check = ArgumentCorrectnessMetric(threshold=0.7)
arg_check.measure(refund_wrong_arg)
print(f"  Score: {arg_check.score:.2f} | Passed: {arg_check.is_successful()}")
print(f"  Only ArgumentCorrectnessMetric catches the order_id contamination.")

Running ToolCorrectnessMetric on the WRONG ARGUMENT case...
  Score: 0.90 | Passed: True
  Reason: process_refund was the correct tool for a refund request. The function selection was appropriate.

ToolCorrectnessMetric PASSED the wrong-argument case (as expected).
This confirms: wrong-tool and wrong-argument are distinct failures requiring distinct metrics.

Now running ArgumentCorrectnessMetric on the same case...
  Score: 0.08 | Passed: False
  Only ArgumentCorrectnessMetric catches the order_id contamination.


## Part 3 — StepEfficiencyMetric: too many tool calls

The `StepEfficiencyMetric` is the tool-calling analog of Module 6's **infinite retrieval loop** failure. Module 6's agent could exhaust `max_hops` without ever being confident it had enough information. This agent can call unnecessary tools before reaching the right one.

The `escalation-01-hardneg` case: customer wants to speak with a manager. The correct single-step path is `escalate_to_human`. The inefficient path calls `check_order_status` and `get_product_info` first (both return errors since no order/product was mentioned), then escalates. Same final output, 3 tool calls instead of 1.

In [4]:
efficiency_metric = StepEfficiencyMetric(threshold=0.7, verbose_mode=True)

# Happy path: direct one-step escalation
escalation_happy = LLMTestCase(
    input="I'm really frustrated with the service I received. I want to speak to a manager.",
    actual_output="I understand your frustration and I'm sorry for the experience you've had. I've escalated your case to a human manager. Your ticket ID is CS-A1B2 and the estimated wait time is about 2 hours.",
    tools_called=[
        ToolCall(
            name="escalate_to_human",
            input_parameters={"reason": "customer requesting to speak with a manager about service dissatisfaction"},
            output='{"ticket_id": "CS-A1B2", "estimated_wait": "2 hours", "queue": "general"}',
        )
    ],
)

# Hard negative: 3 tool calls where 1 was sufficient
escalation_inefficient = LLMTestCase(
    input="I'm really frustrated with the service I received. I want to speak to a manager.",
    # Same output as the happy path — the inefficiency is invisible from the text
    actual_output="I've escalated your case to a human manager. Your ticket ID is CS-C3D4 and the estimated wait time is about 2 hours.",
    tools_called=[
        ToolCall(
            name="check_order_status",   # UNNECESSARY — no order mentioned
            input_parameters={"order_id": "unknown"},
            output='{"error": "Order \'unknown\' not found."}',
        ),
        ToolCall(
            name="get_product_info",     # UNNECESSARY — no product mentioned
            input_parameters={"product_name": "unknown"},
            output='{"error": "Product \'unknown\' not found in catalogue."}',
        ),
        ToolCall(
            name="escalate_to_human",   # CORRECT — but should have been the only call
            input_parameters={"reason": "general complaint"},
            output='{"ticket_id": "CS-C3D4", "estimated_wait": "2 hours", "queue": "general"}',
        ),
    ],
)

print("=== HAPPY PATH: one-step escalation ===")
efficiency_metric.measure(escalation_happy)
print("StepEfficiencyMetric:")
print(f"  Score: {efficiency_metric.score:.2f} | Passed: {efficiency_metric.is_successful()}")
print(f"  Reason: {efficiency_metric.reason}")
print()

print("=== HARD NEGATIVE: 3 tool calls for a 1-step task ===")
efficiency_metric.measure(escalation_inefficient)
print("StepEfficiencyMetric:")
print(f"  Score: {efficiency_metric.score:.2f} | Passed: {efficiency_metric.is_successful()}")
print(f"  Reason: {efficiency_metric.reason}")
print()
print("Key insight: the output is IDENTICAL between the happy path and the hard negative.")
print("Both responses end with: 'Your ticket ID is ..., the estimated wait is about 2 hours'")
print("You cannot detect step inefficiency from the output text alone — you need StepEfficiencyMetric.")

=== HAPPY PATH: one-step escalation ===
StepEfficiencyMetric:
  Score: 0.95 | Passed: True
  Reason: The agent correctly identified that escalation was needed and called escalate_to_human directly without unnecessary intermediate steps.

=== HARD NEGATIVE: 3 tool calls for a 1-step task ===
StepEfficiencyMetric:
  Score: 0.22 | Passed: False
  Reason: The agent made 3 tool calls (check_order_status, get_product_info, escalate_to_human) for a task that required only escalate_to_human. The first two calls were unnecessary — no order ID or product was mentioned in the user's message. This represents significant inefficiency.

Key insight: the output is IDENTICAL between the happy path and the hard negative.
Both responses end with: 'Your ticket ID is ..., the estimated wait is about 2 hours'
You cannot detect step inefficiency from the output text alone — you need StepEfficiencyMetric.


## Part 4 — Load all hard negatives and run all 4 metrics

The `eval_type` field in each golden_dataset.json entry tells you which metric should fail. This is the formal test run.

In [5]:
with open("golden_dataset.json") as f:
    dataset = json.load(f)

hard_negatives = [d for d in dataset if d["is_hard_negative"]]
happy_paths = [d for d in dataset if not d["is_hard_negative"]]

print(f"All {len(dataset)} golden dataset entries loaded.")
print(f"  {len(happy_paths)} happy-path cases")
print(f"  {len(hard_negatives)} hard negatives")
print()
print("Hard negative eval_type distribution:")
from collections import Counter
type_counts = Counter(d["eval_type"] for d in hard_negatives)
for eval_type, count in sorted(type_counts.items()):
    ids = [d["id"] for d in hard_negatives if d["eval_type"] == eval_type]
    print(f"  {eval_type}: {count}  ({', '.join(ids)})")

All 8 golden dataset entries loaded.
  4 happy-path cases
  4 hard negatives

Hard negative eval_type distribution:
  tool_correctness: 2  (order-status-01-hardneg, product-info-01-hardneg)
  argument_correctness: 1  (refund-01-hardneg)
  step_efficiency: 1  (escalation-01-hardneg)


In [6]:
def dataset_entry_to_test_case(entry: dict) -> LLMTestCase:
    tools_called = [
        ToolCall(
            name=tc["name"],
            input_parameters=tc["input_parameters"],
            output=json.dumps(tc["output"]),
        )
        for tc in entry.get("tools_called", [])
    ]
    return LLMTestCase(
        input=entry["user_input"],
        actual_output=entry.get("response", ""),
        tools_called=tools_called,
    )

# Map eval_type to the metric that SHOULD fail
eval_type_to_metric = {
    "tool_correctness":    lambda: ToolCorrectnessMetric(threshold=0.7),
    "argument_correctness": lambda: ArgumentCorrectnessMetric(threshold=0.7),
    "step_efficiency":     lambda: StepEfficiencyMetric(threshold=0.7),
    "task_completion":     lambda: TaskCompletionMetric(
        task="Help customers with order status, refunds, and product questions",
        threshold=0.7
    ),
}

print("Running targeted metric on each hard negative (eval_type → expected failing metric)...")
print()
all_correct = True
for entry in hard_negatives:
    eval_type = entry["eval_type"]
    metric = eval_type_to_metric[eval_type]()
    case = dataset_entry_to_test_case(entry)
    metric.measure(case)
    correctly_failed = not metric.is_successful()
    status = "✓ correctly failed" if correctly_failed else "✗ UNEXPECTED PASS — review this case"
    metric_name = type(metric).__name__
    print(f"  {entry['id']}  [{eval_type}]")
    print(f"    {metric_name}: Score={metric.score:.2f} | Passed={metric.is_successful()}  {status}")
    print()
    if not correctly_failed:
        all_correct = False

if all_correct:
    print("All 4 hard negatives correctly failed their designated metric.")
else:
    print("WARNING: some hard negatives did not fail as expected. Review above.")

Running targeted metric on each hard negative (eval_type → expected failing metric)...

  order-status-01-hardneg  [tool_correctness]
    ToolCorrectnessMetric: Score=0.08 | Passed=False  ✓ correctly failed

  product-info-01-hardneg  [tool_correctness]
    ToolCorrectnessMetric: Score=0.05 | Passed=False  ✓ correctly failed

  refund-01-hardneg  [argument_correctness]
    ArgumentCorrectnessMetric: Score=0.08 | Passed=False  ✓ correctly failed

  escalation-01-hardneg  [step_efficiency]
    StepEfficiencyMetric: Score=0.22 | Passed=False  ✓ correctly failed

All 4 hard negatives correctly failed their designated metric.


## Part 5 — Coverage matrix: the final form

Module 4 Day 4 started this matrix. Module 5 added retrieval columns. Module 6 added agentic columns (premature_stop, reasoning_chain_break). Today's version is the accumulated state with tool-calling failure modes.

The most important thing this matrix does: **make gaps visible**. A '—' cell is a test you haven't written yet.

In [7]:
# Coverage matrix — extended from Module 4 Day 4 through Module 7
capabilities = ["order_management", "product_inquiry", "refund_processing", "escalation"]
failure_modes = ["hallucination", "wrong_tool", "wrong_argument", "inefficient_steps", "ungraceful_failure"]

# Based on golden_dataset.json entries
coverage = {
    "order_management":  {"hallucination": "covered", "wrong_tool": "covered",  "wrong_argument": "covered",  "inefficient_steps": "covered",  "ungraceful_failure": "covered"},
    "product_inquiry":   {"hallucination": "covered", "wrong_tool": "covered",  "wrong_argument": "—",        "inefficient_steps": "—",         "ungraceful_failure": "covered"},
    "refund_processing": {"hallucination": "covered", "wrong_tool": "—",         "wrong_argument": "covered",  "inefficient_steps": "—",         "ungraceful_failure": "covered"},
    "escalation":        {"hallucination": "covered", "wrong_tool": "—",         "wrong_argument": "—",        "inefficient_steps": "covered",  "ungraceful_failure": "covered"},
}

print("=== COVERAGE MATRIX (Module 4 Day 4 → Module 7) ===")
print()

# Header
col_w = 20
header = f"{'Capability':<20}" + "|"
for fm in failure_modes:
    header += f" {fm:<{col_w-1}}|"
print(header)
print("-" * 20 + "|" + ("-" * col_w + "|") * len(failure_modes))

# Rows
for cap in capabilities:
    row = f"{cap:<20}|"
    for fm in failure_modes:
        cell = coverage[cap][fm]
        row += f" {cell:<{col_w-1}}|"
    print(row)

print()
print("Key:")
print("  covered = at least one test case (happy-path or hard negative) in golden_dataset.json")
print("  — = gap — no test case covers this cell yet")
print()
print("The matrix is not complete — this is intentional.")
print("Gaps (—) are the exercise: which cell would you add next, and what would the hard negative look like?")
print("Module 4 Day 4's discipline: the matrix makes the gap visible. Writing the test is the response.")
print()
print("Metrics responsible for each column:")
print(f"  {'hallucination':<20}→ TaskCompletionMetric (catches answers that contradict tool output)")
print(f"  {'wrong_tool':<20}→ ToolCorrectnessMetric (inspects tools_called names)")
print(f"  {'wrong_argument':<20}→ ArgumentCorrectnessMetric (inspects tools_called input_parameters)")
print(f"  {'inefficient_steps':<20}→ StepEfficiencyMetric (counts tool calls vs minimum needed)")
print(f"  {'ungraceful_failure':<20}→ TaskCompletionMetric (catches 'task not completed' on unknowable inputs)")

=== COVERAGE MATRIX (Module 4 Day 4 → Module 7) ===

Capability          | hallucination    | wrong_tool          | wrong_argument      | inefficient_steps   | ungraceful_failure  
--------------------|------------------|---------------------|---------------------|---------------------|---------------------
order_management    | covered          | covered             | covered             | covered             | covered             
product_inquiry     | covered          | covered             | —                   | —                   | covered             
refund_processing   | covered          | —                   | covered             | —                   | covered             
escalation          | covered          | —                   | —                   | covered             | covered             

Key:
  covered = at least one test case (happy-path or hard negative) in golden_dataset.json
  — = gap — no test case covers this cell yet

The matrix is not complete — this is i

## Part 6 — Retiring Module 6's hand-rolled judges

Module 6 built three hand-rolled verdict classes. Here's the formal retirement — showing that the DeepEval equivalents are more robust (LLM-judged, not string-matched).

In [8]:
print("=== Module 6 hand-rolled judges → Module 7 DeepEval replacements ===")
print()

retirement_map = [
    {
        "old": "GracefulFailureVerdict (Module 6)",
        "old_method": "checked for hedge phrases ['don't have', 'unable to find', ...] in output text",
        "old_problem": "brittle — 'I regret I cannot provide...' would PASS the phrase check if 'cannot' wasn't in the list",
        "new": "TaskCompletionMetric",
        "new_advantage": "phrasing-agnostic, scores on a 0-1 scale, threshold-able",
        "new_question": "'was the task completed?'",
    },
    {
        "old": "ReasoningChainVerdict (Module 6)",
        "old_method": "must_not_include=['$50'] string check — if agent output contained $50, verdict=FAIL",
        "old_problem": "brittle — 'The fee is not $50, it is $0' would FAIL even though it's correct",
        "new": "ToolCorrectnessMetric",
        "new_advantage": "catches wrong tool regardless of how the output is phrased",
        "new_question": "inspects ToolCall records, not output strings",
    },
    {
        "old": "MemoryRetentionVerdict (Module 6)",
        "old_method": "must_include=['WidgetPro 3000'] string check — did the product name survive to the answer?",
        "old_problem": "brittle — 'WidgetPro3000' (no space) would FAIL must_include('WidgetPro 3000')",
        "new": "ArgumentCorrectnessMetric",
        "new_advantage": "LLM understands that 'order 12345' and 'order_id=12345' refer to the same thing",
        "new_question": "LLM judge asking 'were the right arguments passed?'",
    },
]

for i, item in enumerate(retirement_map, 1):
    print(f"{i}. {item['old']}")
    print(f"   Old: {item['old_method']}")
    print(f"   Old PROBLEM: {item['old_problem']}")
    print(f"   New: {item['new']} — {item['new_question']}")
    print(f"   New ADVANTAGE: {item['new_advantage']}")
    print()

print("All three Module 6 verdict classes are now retired.")
print("The DeepEval equivalents are imported in this notebook and running above.")

=== Module 6 hand-rolled judges → Module 7 DeepEval replacements ===

1. GracefulFailureVerdict (Module 6)
   Old: checked for hedge phrases ['don't have', 'unable to find', ...] in output text
   Old PROBLEM: brittle — 'I regret I cannot provide...' would PASS the phrase check if 'cannot' wasn't in the list
   New: TaskCompletionMetric — LLM judge asking 'was the task completed?'
   New ADVANTAGE: phrasing-agnostic, scores on a 0-1 scale, threshold-able

2. ReasoningChainVerdict (Module 6)
   Old: must_not_include=['$50'] string check — if agent output contained $50, verdict=FAIL
   Old PROBLEM: brittle — 'The fee is not $50, it is $0' would FAIL even though it's correct
   New: ToolCorrectnessMetric — inspects ToolCall records, not output strings
   New ADVANTAGE: catches wrong tool regardless of how the output is phrased

3. MemoryRetentionVerdict (Module 6)
   Old: must_include=['WidgetPro 3000'] string check — did the product name survive to the answer?
   Old PROBLEM: brittle — '

## Part 7 — Live agent run + full 4-metric evaluation

In [9]:
print("Live agent run — refund request for order #67890")
live_response = asyncio.run(
    run_agent("I'd like to request a refund for order #67890. The product arrived damaged.", verbose=True)
)

print("Agent output:")
print(f"  {live_response.output}")
print()

live_case = LLMTestCase(
    input="I'd like to request a refund for order #67890. The product arrived damaged.",
    actual_output=live_response.output,
    tools_called=to_deepeval_tool_calls(live_response),
)

metrics = [
    ("TaskCompletionMetric",     TaskCompletionMetric(task="Help customers with order status, refunds, and product questions", threshold=0.7)),
    ("ToolCorrectnessMetric",    ToolCorrectnessMetric(threshold=0.7)),
    ("ArgumentCorrectnessMetric", ArgumentCorrectnessMetric(threshold=0.7)),
    ("StepEfficiencyMetric",     StepEfficiencyMetric(threshold=0.7)),
]

print("Running all 4 metrics...")
all_passed = True
for name, metric in metrics:
    metric.measure(live_case)
    passed = metric.is_successful()
    print(f"  {name + ':':<26} Score={metric.score:.2f} | Passed={passed}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print("All 4 metrics passed on the live happy-path response.")
else:
    print("One or more metrics failed on the live response — review above.")

Live agent run — refund request for order #67890
[tool call] process_refund({'order_id': '67890', 'reason': 'product arrived damaged'})
[tool result] {'refund_id': 'R-A1B2C3', 'amount': 299.0, 'status': 'initiated', 'reason': 'product arrived damaged'}

Agent output:
  I've initiated a refund for your TurboMax Pro order #67890 in the amount of $299.00. Your refund ID is R-A1B2C3. You should see the funds returned within 5-7 business days. I apologize for the damaged product experience.

Running all 4 metrics...
  TaskCompletionMetric:     Score=0.94 | Passed=True
  ToolCorrectnessMetric:    Score=0.95 | Passed=True
  ArgumentCorrectnessMetric: Score=0.96 | Passed=True
  StepEfficiencyMetric:     Score=0.97 | Passed=True

All 4 metrics passed on the live happy-path response.


## Summary — Module 7 complete

| Module 6 hand-rolled | Module 7 DeepEval metric | What it catches |
|---|---|---|
| `GracefulFailureVerdict` | `TaskCompletionMetric` | Task not completed, graceful admission of unknowns |
| `ReasoningChainVerdict` | `ToolCorrectnessMetric` | Wrong tool called, no tool called |
| `MemoryRetentionVerdict` | `ArgumentCorrectnessMetric` | Right tool, wrong arguments (context contamination) |
| *(new in Module 7)* | `StepEfficiencyMetric` | Too many tool calls for the task |

**Module 4 Day 4 techniques used in this module:**
- Equivalence partitioning → `tool_selection_partitions` (Day 1)
- Boundary value analysis → `order_id` boundary (current message vs. context history)
- Hard negatives → `tools_called=[]`, wrong tool, wrong arg, inefficient path
- Coverage matrix → extended with `wrong_tool`, `wrong_argument`, `inefficient_steps` columns

**Next:** Module 8 — Adversarial Testing & Red-Teaming. Same `LLMTestCase` format, same agent. The `input` field now contains adversarial probes.